In [ ]:
import os
import glob
import pandas as pd


def merge_breakpoint_csvs(
    input_path: str,
    output_dir: str,
    output_filename: str,
    required_columns: list[str] | None = None
) -> str:
    """
    Busca todos los archivos .csv dentro de `input_path`, extrae columnas específicas,
    agrega una columna 'file' con el nombre del archivo, concatena todo en un solo DataFrame
    y lo guarda en `output_dir` con el nombre `output_filename`.

    Parameters
    ----------
    input_path : str
        Ruta donde están los archivos .csv de entrada.
    output_dir : str
        Ruta donde se guardará el nuevo archivo consolidado.
    output_filename : str
        Nombre del archivo de salida, por ejemplo: 'breakpoints_unificados.csv'
    required_columns : list[str] | None
        Lista de columnas a copiar. Si no se especifica, usa las columnas esperadas.

    Returns
    -------
    str
        Ruta completa del archivo generado.
    """
    if required_columns is None:
        required_columns = [
            "ID",
            "breakpoint_1 (sec)",
            "breakpoint_2 (sec)",
            "breakpoint_1 (vp)",
            "breakpoint_2 (vp)",
        ]

    if not os.path.exists(input_path):
        raise FileNotFoundError(f"La ruta de entrada no existe: {input_path}")

    os.makedirs(output_dir, exist_ok=True)

    if not output_filename.lower().endswith(".csv"):
        output_filename += ".csv"

    csv_files = glob.glob(os.path.join(input_path, "*.csv"))

    if not csv_files:
        raise FileNotFoundError(f"No se encontraron archivos .csv en: {input_path}")

    merged_data = []

    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file)

            missing_cols = [col for col in required_columns if col not in df.columns]
            if missing_cols:
                print(f"[WARNING] Se omitió '{csv_file}' porque le faltan columnas: {missing_cols}")
                continue

            temp_df = df[required_columns].copy()
            temp_df.insert(0, "file", os.path.basename(csv_file))
            merged_data.append(temp_df)

        except Exception as e:
            print(f"[ERROR] No se pudo procesar '{csv_file}': {e}")

    if not merged_data:
        raise ValueError("No se pudo generar el archivo final porque ningún CSV tenía las columnas requeridas.")

    final_df = pd.concat(merged_data, ignore_index=True)

    output_path = os.path.join(output_dir, output_filename)
    final_df.to_csv(output_path, index=False, encoding="utf-8-sig")

    print(f"\nArchivo guardado en:\n{output_path}")
    return output_path


In [ ]:

# =========================
# Ejemplo de uso
# =========================
if __name__ == "__main__":
    input_path = r"C:\Users\Mariana\Documents\freshwater_lens\results\dgh_breakpoints"
    output_dir = r"C:\Users\Mariana\Documents\freshwater_lens\results\dgh_breakpoints"
    output_filename = "bp_experiments_lrs70.csv"

    merge_breakpoint_csvs(
        input_path=input_path,
        output_dir=output_dir,
        output_filename=output_filename
    )